# Build the 128 px and 256 px caches in Google Colab (Colab Pro + Drive)

**First session only (~45–90 min):** unpack `train-tif-v2.tar.7z` from your Drive → build **both caches in one pass** → save them to Drive.
**Every later session (~5–15 min):** run section 8 only, to copy the caches back from Drive.

| Folder | Contents | Size |
|---|---|---|
| `amazon_tif_128/cache` | 128 × 128 chips, 5 channels (R, G, B, NIR, NDVI), same format as the original cache | ~6.6 GB |
| `amazon_tif_256/cache` | 256 × 256 chips, full resolution, same channels | ~27 GB |

Both caches share the same chip order, train/val/test split and band scaling, so a model can switch between them mid-training (for example, progressive resizing).

**Before you start:** Runtime → Change runtime type → choose **A100** (or L4) and turn on **High-RAM**.

## 1. Check the machine

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv   # A100 or L4 recommended
!free -h                                                  # High-RAM shows ~50 GB or more
!df -h /content                                           # needs ~60 GB free for the TIFs + both caches

/bin/bash: line 1: nvidia-smi: command not found
               total        used        free      shared  buff/cache   available
Mem:            50Gi       1.8Gi        47Gi       2.2Mi       2.8Gi        49Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
overlay         226G   85M  226G   1% /


## 2. Mount Google Drive and set the paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ARCHIVE   = "/content/drive/MyDrive/train-tif-v2.tar.7z"     # where you uploaded the archive (edit if in a subfolder)
CSV_DRIVE = "/content/drive/MyDrive/train_v2.csv"            # optional: if you uploaded the labels CSV too
DRIVE_DIR = "/content/drive/MyDrive/amazon_caches"           # where the finished caches will be saved

RAW     = "/content/raw"                                     # unpacked TIFs go on Colab's fast local disk
ROOT128 = "/content/amazon_tif_128"
ROOT256 = "/content/amazon_tif_256"

import os
print("archive found:", os.path.exists(ARCHIVE), "| labels CSV in Drive:", os.path.exists(CSV_DRIVE))

Mounted at /content/drive
archive found: True | labels CSV in Drive: True


## 3. Unpack the TIFs from Drive (~15–30 min)
The archive is streamed straight from Drive: 7-Zip decompresses to its output and `tar` extracts on the fly. The archive stays in your Drive, and the ~21 GB intermediate `.tar` is never written. If this stalls, copy the archive first (`!cp "{ARCHIVE}" /content/`) and unpack from `/content`.

In [ ]:
!apt-get -qq install -y p7zip-full > /dev/null           # 7-Zip for Linux
!mkdir -p {RAW}
!7z x -so "{ARCHIVE}" | tar xf - -C {RAW}
!find {RAW} -name "*.tif" | wc -l                        # should print 40479

40479


## 4. Get the labels file `train_v2.csv`
It isn't inside the TIF archive. Use the copy in your Drive if you uploaded one. Otherwise run the Kaggle cell (accept the competition rules on kaggle.com first, and have `kaggle.json` from Settings → API).

In [ ]:
if os.path.exists(CSV_DRIVE):
    !cp "{CSV_DRIVE}" {RAW}/
    print("copied train_v2.csv from Drive")
else:
    print("No CSV in Drive. Run the Kaggle cell below.")

copied train_v2.csv from Drive


In [ ]:
!ls -la {RAW} | head

total 2756
drwxr-xr-x 3 root root    4096 Nov 11 00:42 .
drwxr-xr-x 1 root root    4096 Nov 11 00:39 ..
drwxrwxr-x 2 root root 1380352 May  1  2017 train-tif-v2
-rw------- 1 root root 1432177 Nov 11 00:42 train_v2.csv


## 5. Write the cache builder

In [ ]:
!pip -q install tifffile                                   # TIF reader (opencv, pandas, sklearn are preinstalled)

In [ ]:
%%writefile build_caches.py
#!/usr/bin/env python3
"""
Build the 256 px and 128 px caches in ONE pass over the TIFs.

Each TIF is read once, normalized exactly like run_matched.py does, and then:
  * written as-is to the 256 px cache (the TIFs are already 256 x 256), and
  * shrunk with cv2.INTER_AREA and written to the 128 px cache.
Because both caches come from the same pass, they share the same chip order, the same
train/val/test split and the same band scaling, so a model can switch between them mid-training.

The output format is identical to run_matched.py's own cache, so run_matched.py can use
either folder directly (--root <folder>, --tile 128 or 256).

With --leak-free, the band order and percentile bounds come from training chips only
(the original pipeline used 600 chips from all data, including test chips).

Usage:
    python build_caches.py --raw /content/raw --root128 /content/amazon_tif_128 --root256 /content/amazon_tif_256 --leak-free
"""
import argparse, glob, json, os, sys, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import cv2
import tifffile
from sklearn.model_selection import train_test_split

ap = argparse.ArgumentParser()
ap.add_argument("--raw", required=True, help="folder with train-tif-v2/*.tif and train_v2.csv (searched recursively)")
ap.add_argument("--root128", required=True, help="project folder for the 128 px cache (cache/ is created inside)")
ap.add_argument("--root256", required=True, help="project folder for the 256 px cache (cache/ is created inside)")
ap.add_argument("--rgb-only-256", action="store_true",
                help="store only R, G, B in the 256 px cache (~16 GB instead of ~27 GB). "
                     "Fine for RGB models such as C; NIR/NDVI models then need the 128 px cache.")
ap.add_argument("--leak-free", action="store_true",
                help="compute the band order and 1st/99th percentile bounds from TRAINING chips only "
                     "(fixes the minor leakage in the original pipeline). Without it, the caches match the original exactly.")
ap.add_argument("--threads", type=int, default=8)
args = ap.parse_args()

SEED = 42                      # same seed as run_matched.py, so splits and samples match the original runs
t0 = time.time()

# ------------------------------------------------------------------ 1. find the TIFs and labels (same rules as run_matched.py)
tifs = sorted(glob.glob(f"{args.raw}/**/*.tif", recursive=True))
csvs = [p for p in glob.glob(f"{args.raw}/**/train_v2.csv", recursive=True) if os.path.isfile(p)]
if len(tifs) < 100 or not csvs:
    sys.exit(f"Found {len(tifs)} TIFs and {len(csvs)} train_v2.csv under {args.raw}. Unpack the data first.")
tifDir = os.path.dirname(tifs[0])
df = pd.read_csv(csvs[0])
df = df[df["image_name"].map(lambda n: os.path.exists(os.path.join(tifDir, n + ".tif")))].reset_index(drop=True)

tagLists = df["tags"].str.split()
labels = sorted({t for tl in tagLists for t in tl})            # alphabetical, as in run_matched.py
y = np.zeros((len(df), len(labels)), dtype=np.uint8)            # multi-hot label matrix
for i, tl in enumerate(tagLists):
    for t in tl:
        y[i, labels.index(t)] = 1
names = df["image_name"].tolist()
nTiles = len(names)
print(f"{nTiles} chips with a TIF, {len(labels)} labels")

def read_tif(n):
    a = tifffile.imread(os.path.join(tifDir, n + ".tif")).astype(np.float32)
    if a.ndim == 3 and a.shape[0] == 4 and a.shape[-1] != 4:   # some readers return (bands, H, W)
        a = np.transpose(a, (1, 2, 0))
    return a

# ------------------------------------------------------------------ 2. split first (it only needs labels), then band order + bounds
# The split is identical to run_matched.py: stratified by each chip's rarest label, 70/15/15, seed 42.
freq = y.sum(0)
strata = np.array([min(np.where(row == 1)[0], key=lambda j: freq[j]) for row in y])
idx = np.arange(nTiles)
tr, tmp = train_test_split(idx, test_size=0.30, stratify=strata, random_state=SEED)
va, te = train_test_split(tmp, test_size=0.50, stratify=strata[tmp], random_state=SEED)

def band_stats(sampleIdx):
    """Band order + 1st/99th percentiles from a sample of chips (same method as run_matched.py)."""
    raw = np.stack([read_tif(names[i]) for i in sampleIdx])
    ys = y[sampleIdx]
    prim = ys[:, labels.index("primary")] == 1
    open_ = (ys[:, labels.index("agriculture")] == 1) | (ys[:, labels.index("bare_ground")] == 1)
    prim, open_ = prim & ~open_, open_ & ~prim
    tileMean = raw.reshape(len(sampleIdx), -1, 4).mean(1)
    ratio = tileMean[open_].mean(0) / tileMean[prim].mean(0)    # open land vs forest brightness per band
    order = "RGBN" if ratio[0] > 1.15 * ratio[2] else "BGRN"
    return order, np.percentile(raw.reshape(-1, 4), 1, axis=0), np.percentile(raw.reshape(-1, 4), 99, axis=0)

rng = np.random.RandomState(SEED)                               # same random sequence as the original pipeline
sampAll = rng.choice(nTiles, min(600, nTiles), replace=False)   # original: 600 chips from ALL data (incl. test)
orderAll, loAll, hiAll = band_stats(sampAll)
if args.leak_free:
    sampTr = np.random.RandomState(SEED + 1).choice(tr, min(600, len(tr)), replace=False)   # training chips only
    bandOrder, lo, hi = band_stats(sampTr)
    # How much did the leakage actually change the scaling? (useful number for the presentation)
    rel = np.abs(np.concatenate([lo - loAll, hi - hiAll])) / (np.concatenate([hiAll - loAll] * 2) + 1e-6)
    print(f"Leak-free bounds differ from the all-data bounds by at most {rel.max():.2%} of each band's range"
          f" (band order: {bandOrder} vs {orderAll})")
else:
    bandOrder, lo, hi = orderAll, loAll, hiAll
bandMap = {"BGRN": dict(b=0, g=1, r=2, nir=3), "RGBN": dict(b=2, g=1, r=0, nir=3)}[bandOrder]
print("Band order:", bandOrder, "| bounds from", "training chips only" if args.leak_free else "all chips (original method)")

# ------------------------------------------------------------------ 3. one pass: normalize each chip, write both caches
caches = {128: os.path.join(args.root128, "cache"), 256: os.path.join(args.root256, "cache")}
nCh = {128: 5, 256: 3 if args.rgb_only_256 else 5}
chanNames = ["R", "G", "B", "NIR", "NDVI"]
for size, c in caches.items():
    os.makedirs(c, exist_ok=True)
    os.makedirs(os.path.join(os.path.dirname(c), "runs"), exist_ok=True)
    os.makedirs(os.path.join(os.path.dirname(c), "figs"), exist_ok=True)
X = {size: np.lib.format.open_memmap(os.path.join(c, "X.npy"), mode="w+", dtype=np.float16,
                                     shape=(nTiles, nCh[size], size, size))
     for size, c in caches.items()}                             # written straight to disk, so RAM stays low

def process(i):
    a = read_tif(names[i])
    red, nir = a[..., bandMap["r"]], a[..., bandMap["nir"]]
    ndvi = (nir - red) / (nir + red + 1e-6)                     # NDVI from raw values, before scaling
    chans = [np.clip((a[..., src] - lo[src]) / (hi[src] - lo[src] + 1e-6), 0, 1)
             for src in (bandMap["r"], bandMap["g"], bandMap["b"], bandMap["nir"])]
    chans.append((np.clip(ndvi, -1, 1) + 1) / 2)                # NDVI rescaled from [-1, 1] to [0, 1]
    full = np.stack(chans, -1).astype(np.float32)               # (256, 256, 5)
    if full.shape[0] != 256 or full.shape[1] != 256:
        full = cv2.resize(full, (256, 256), interpolation=cv2.INTER_AREA)
    X[256][i] = np.transpose(full[..., :nCh[256]], (2, 0, 1)).astype(np.float16)          # full resolution
    small = cv2.resize(full, (128, 128), interpolation=cv2.INTER_AREA)                     # same resize as before
    X[128][i] = np.transpose(small, (2, 0, 1)).astype(np.float16)

with ThreadPoolExecutor(args.threads) as ex:
    for k, _ in enumerate(ex.map(process, range(nTiles))):
        if k % 2000 == 0:
            print(f"  {k}/{nTiles} chips  [{time.time() - t0:.0f}s]")
for m in X.values():
    m.flush()

# ------------------------------------------------------------------ 4. save split, labels and per-cache stats
pick = np.sort(rng.choice(np.sort(tr), min(3000, len(tr)), replace=False))           # training chips for mean/std

for size, c in caches.items():
    np.save(os.path.join(c, "Y.npy"), y)
    json.dump(labels, open(os.path.join(c, "labels.json"), "w"))
    json.dump(names, open(os.path.join(c, "names.json"), "w"))
    json.dump({"order": bandOrder, **bandMap}, open(os.path.join(c, "order.json"), "w"))
    json.dump({"lo": lo.tolist(), "hi": hi.tolist(), "leak_free": bool(args.leak_free),
               "lo_alldata": loAll.tolist(), "hi_alldata": hiAll.tolist()},
              open(os.path.join(c, "band_stats.json"), "w"))
    np.savez(os.path.join(c, "splits.npz"), train=np.sort(tr), val=np.sort(va), test=np.sort(te))
    sub = np.asarray(np.load(os.path.join(c, "X.npy"), mmap_mode="r")[pick]).astype(np.float32)
    np.savez(os.path.join(c, "stats.npz"), mean=sub.mean((0, 2, 3)), std=sub.std((0, 2, 3)) + 1e-6)
    json.dump({"n": nTiles, "tile": size, "jpg_mode": False, "channels": chanNames[:nCh[size]]},
              open(os.path.join(c, "cache_meta.json"), "w"))     # run_matched.py checks this file
    gb = os.path.getsize(os.path.join(c, "X.npy")) / 1e9
    print(f"{size} px cache: {c}  ({nCh[size]} channels, {gb:.1f} GB)")

print(f"Done in {(time.time() - t0) / 60:.1f} min. train/val/test = {len(tr)}/{len(va)}/{len(te)}")


Writing build_caches.py


## 6. Build both caches (~20–40 min)

**`LEAK_FREE`:** the original pipeline chose the band order and the 1st/99th percentile scaling from 600 chips drawn from *all* data, test chips included. That's a minor form of data leakage.
- `LEAK_FREE = True` (recommended): uses training chips only. The builder also prints how much the scaling actually changed, a useful number for your limitations slide. Because the scaling may differ very slightly from the original cache, **retrain the baselines you compare against (A, C, D, E) on this cache** so every comparison uses the same data. That's ~15–25 min on an A100.
- `LEAK_FREE = False`: reproduces the original cache exactly, so results from your teammate's earlier runs stay directly comparable.

In [ ]:
import sys, subprocess

LEAK_FREE = True

cmd = [sys.executable, "-u", "build_caches.py", "--raw", RAW, "--root128", ROOT128, "--root256", ROOT256,
       "--threads", "8"]
if LEAK_FREE:
    cmd.append("--leak-free")
print(" ".join(cmd))
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:          # progress prints every 2,000 chips
    print(line, end="")
p.wait(); print("exit code:", p.returncode)

/usr/bin/python3 -u build_caches.py --raw /content/raw --root128 /content/amazon_tif_128 --root256 /content/amazon_tif_256 --threads 8 --leak-free
40479 chips with a TIF, 17 labels
Leak-free bounds differ from the all-data bounds by at most 11.42% of each band's range (band order: BGRN vs BGRN)
Band order: BGRN | bounds from training chips only
  0/40479 chips  [14s]
  2000/40479 chips  [23s]
  4000/40479 chips  [32s]
  6000/40479 chips  [42s]
  8000/40479 chips  [54s]
  10000/40479 chips  [63s]
  12000/40479 chips  [74s]
  14000/40479 chips  [85s]
  16000/40479 chips  [95s]
  18000/40479 chips  [106s]
  20000/40479 chips  [117s]
  22000/40479 chips  [128s]
  24000/40479 chips  [138s]
  26000/40479 chips  [149s]
  28000/40479 chips  [160s]
  30000/40479 chips  [173s]
  32000/40479 chips  [183s]
  34000/40479 chips  [193s]
  36000/40479 chips  [205s]
  38000/40479 chips  [216s]
  40000/40479 chips  [227s]
128 px cache: /content/amazon_tif_128/cache  (5 channels, 6.6 GB)
256 px cache: /c

### Check that the two caches line up
Row *i* must be the same chip in both caches, with the same split and scaling. The last line confirms that shrinking the 256 px chips reproduces the 128 px cache (differences around 0.001 are float16 rounding).

In [ ]:
import numpy as np, json
c128, c256 = f"{ROOT128}/cache", f"{ROOT256}/cache"
for f in ["names.json", "band_stats.json", "order.json", "labels.json"]:
    print(f"{f:<16} identical:", open(f"{c128}/{f}").read() == open(f"{c256}/{f}").read())
s1, s2 = np.load(f"{c128}/splits.npz"), np.load(f"{c256}/splits.npz")
print("splits.npz       identical:", all((s1[k] == s2[k]).all() for k in s1.files))
print("leak-free scaling:", json.load(open(f"{c128}/band_stats.json"))["leak_free"])

x128 = np.load(f"{c128}/X.npy", mmap_mode="r"); x256 = np.load(f"{c256}/X.npy", mmap_mode="r")
print("shapes:", x128.shape, x256.shape)
rows = np.arange(0, len(x128), 4000)                       # spot-check 11 chips
k = x256.shape[1]
a = np.asarray(x256[rows]).astype(np.float32)
pooled = a.reshape(len(rows), k, 128, 2, 128, 2).mean((3, 5))    # 2x2 average = 256 -> 128
print("max |pooled 256 - 128|:", float(np.abs(pooled - np.asarray(x128[rows, :k]).astype(np.float32)).max()))

names.json       identical: True
band_stats.json  identical: True
order.json       identical: True
labels.json      identical: True
splits.npz       identical: True
leak-free scaling: True
shapes: (40479, 5, 128, 128) (40479, 5, 256, 256)
max |pooled 256 - 128|: 0.0003662109375


## 7. Save both caches to Drive (~34 GB, ~10–20 min)

In [ ]:
!mkdir -p "{DRIVE_DIR}"
!rsync -ah --info=progress2 {ROOT256} "{DRIVE_DIR}/"      # shows progress; resumes if interrupted
!rsync -ah --info=progress2 {ROOT128} "{DRIVE_DIR}/"

from google.colab import drive
drive.flush_and_unmount()                                # waits until everything is written to Drive
print("Drive sync finished")
drive.mount("/content/drive")
!ls -la "{DRIVE_DIR}"/amazon_tif_256/cache/X.npy "{DRIVE_DIR}"/amazon_tif_128/cache/X.npy
# Optional once this succeeds: the archive in Drive is no longer needed (it can be re-downloaded from Kaggle)

         26.53G 100%  124.20MB/s    0:03:23 (xfr#9, to-chk=0/13)
          6.63G 100%   71.53MB/s    0:01:28 (xfr#9, to-chk=0/13)
Drive sync finished
Mounted at /content/drive
-rw------- 1 root root  6632079488 Nov 11 00:46 /content/drive/MyDrive/amazon_caches/amazon_tif_128/cache/X.npy
-rw------- 1 root root 26528317568 Nov 11 00:46 /content/drive/MyDrive/amazon_caches/amazon_tif_256/cache/X.npy


## 8. Later sessions: restore the caches (run instead of sections 3–7)
Copy the caches to Colab's local disk rather than training from Drive directly. Training reads the cache constantly, and Drive is much slower.

In [ ]:
# from google.colab import drive; drive.mount("/content/drive")
# DRIVE_DIR = "/content/drive/MyDrive/amazon_caches"
# !cp -r "{DRIVE_DIR}/amazon_tif_128" /content/            # ~6.6 GB
# !cp -r "{DRIVE_DIR}/amazon_tif_256" /content/            # ~27 GB, only when you need 256 px

## 9. Using the caches with `run_matched.py`
- **128 px:** `--root /content/amazon_tif_128 --tile 128`; **256 px:** `--root /content/amazon_tif_256 --tile 256`.
- Apply the **section-1 patch** to `run_matched.py` (load labels and names from the cache), so it runs without the TIFs in later sessions.
- **Save `runs/` to Drive after training** (`!cp -r /content/amazon_tif_128/runs "{DRIVE_DIR}/amazon_tif_128/"`). Checkpoints and predictions in `/content` disappear with the session.
- With High-RAM (~50 GB), the 256 px RGB splits (~16 GB) fit in memory. For 5-channel 256 px runs (~27 GB), High-RAM is enough too, but the disk-based loader is safer.